# 🔬 STRATIFIED 5-FOLD CROSS-VALIDATION ĐA PHƯƠNG THỨC (2D + 3D CEUS) - VERSION 2
### *Quy Trình Hai Giai Đoạn: YOLO11 Sàng Lọc Tổn Thương + EfficientNet-B3 Phân Loại 7 Lớp Bệnh Học*
---
**Quy chuẩn thực nghiệm & Kiểm soát phương pháp luận y khoa (Version 2):**
1. **Quản lý Checkpoint & Cache Dữ Liệu**: Thư mục kết quả trên Google Drive và tên file được đánh dấu phiên bản (`v2_{inpaint|noinpaint}`) để ngăn chặn việc tái sử dụng im lặng các mô hình cũ. Tách riêng thư mục cache ảnh trên Colab theo biến thể inpainting. Xác thực cấu trúc 26 cột bắt buộc trước khi nạp lại file dự đoán OOF.
2. **Khử Chú Thích Có Kiểm Soát (OCR-Guided Inpainting)**: Tích hợp EasyOCR song ngữ Anh - Trung (`['en', 'ch_sim']`) để nhận diện chính xác các văn bản máy in. Tách biệt hoàn toàn luật xử lý: ảnh 2D dùng bộ lọc màu vàng HSV; ảnh 3D CEUS (tông cam/đồng) dùng ngưỡng trắng thuần kết hợp phân tích hình thái nét mảnh (không áp dụng luật màu vàng để tránh xóa nhầm mô tưới máu). Tính `mask_ratio` và gắn cờ các ca có nguy cơ can thiệp vào u thật.
3. **Huấn Luyện & Đánh Giá Không Rò Rỉ**: YOLO11n huấn luyện với `val=False`, cố định 20 epochs và chỉ lấy checkpoint cuối cùng (`last.pt`). Lưu độ tin cậy thực `top_box_conf` ở ngưỡng quét `conf=0.05` (không ép về 0 khi dưới 0.30) để vẽ đường cong ROC/PR của Stage 1. Lưu toàn bộ vector xác suất 7 lớp cho cả YOLO crop và GT crop (chọn box GT lớn nhất khi có nhiều u). Chuẩn bị sẵn biến thể Stage 2 với độ lệch box `BOX_JITTER = False`.
4. **Báo Cáo Thống Kê Toàn Diện**: Khôi phục bảng sàng lọc OOF gộp (3 hàng: 2D, CEUS, Tổng) kèm 95% Wilson CI và bảng định vị tổn thương ($	ext{IoU} \ge 0.50$). Báo cáo OOF Macro-F1, Cascade Degradation Gap kèm Bootstrap 95% CI ($B=1.000$) và Mean ± SD theo fold. Phân rã 7 lớp theo phương thức 2D vs CEUS, phân tích False Negative (bỏ sót) theo lớp, phân tích ROC-AUC Ác tính (GT crop vs YOLO crop end-to-end) kèm quét ngưỡng 0.1–0.7, đo lường độ hiệu chuẩn mô hình (ECE, Reliability Diagram), và ma trận nhầm lẫn 7×8 hiển thị cỡ mẫu từng hàng.
5. **Explainable AI (Grad-CAM) Chuẩn Xác**: Chọn ca có chủ đích (TP Ác tính, TP Teratoma, FP Buồng trứng thường, FN Bỏ sót, Misclassified, High mask ratio). Nạp đúng mô hình EfficientNet-B3 của fold mà ca đó đóng vai trò kiểm định độc lập. Khắc phục lỗi hiển thị màu sắc RGB. Trực quan hóa 4 cột và định lượng tỉ lệ năng lượng heatmap rơi vào mask chú thích ($E_{	ext{mask}}$) và vùng u thật ($E_{	ext{tumor}}$).
6. **Chia Fold & Tái Lập Bằng Perceptual Hash (pHash)**: Gom nhóm các ảnh gần trùng bằng 64-bit DCT pHash (khoảng cách Hamming $\le 4$) và phân tầng bằng `StratifiedGroupKFold`. Lưu manifest chia fold `kfold_manifest_v2.csv` và metadata thực nghiệm lên Google Drive.


## 1. Cài Đặt Môi Trường, Thiết Lập Seed & Kiểm Tra GPU (NVIDIA T4)


In [ ]:
!pip install -q ultralytics torchvision timm opencv-python-headless pandas matplotlib seaborn tabulate scikit-learn easyocr

import os, sys, random, shutil, json, math, time
from pathlib import Path
from collections import Counter, defaultdict
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from ultralytics import YOLO
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, precision_recall_curve, roc_curve, auc

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)

print("=" * 68)
print("🎮 KIỂM TRA PHẦN CỨNG & THIẾT BỊ TÍNH TOÁN GOOGLE COLAB")
print("=" * 68)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {device}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Dung lượng VRAM: {vram:.2f} GB")
else:
    print("⚠️ Cảnh báo: Vui lòng vào Runtime > Change runtime type > Chọn GPU T4!")
print("=" * 68)


## 2. Kết Nối Google Drive, Cấu Hình Phiên Bản & Chuẩn Bị Bộ Dữ Liệu MMOTU (2D & 3D CEUS)


In [ ]:
from google.colab import drive
import zipfile, os, shutil
from pathlib import Path

# 1. Kết nối Google Drive an toàn
drive_mounted = False
try:
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    drive_mounted = True
    print("✅ Đã kết nối thành công Google Drive!")
except Exception as e:
    print(f"⚠️ Chưa thể kết nối Google Drive ({e}).")

# 2. Cấu hình biến thể và đường dẫn thư mục lưu trữ (Chống nạp đè checkpoint cũ)
VERSION_TAG = "v2"
APPLY_INPAINTING = True      # Cấu hình: True (có khử chữ/caliper) hoặc False (chạy đối chứng)
BOX_JITTER = False            # Cấu hình: True (huấn luyện Stage 2 với jitter ±10%) hoặc False
RUN_ABLATION = False          # Cấu hình: True (chạy thử nghiệm ablation ở Cell 16)

DIR_SUFFIX = "inpaint" if APPLY_INPAINTING else "noinpaint"
DRIVE_DIR = Path(f'/content/drive/MyDrive/NCKH_5Fold_{VERSION_TAG}_{DIR_SUFFIX}')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Thư mục lưu kết quả trên Google Drive: {DRIVE_DIR}")

# 3. Chuẩn bị bộ dữ liệu MMOTU (2D & 3D CEUS)
# Đích sao chép trên ổ SSD cục bộ của Colab để tối ưu tốc độ đọc GPU
extract_dest = Path('/content/dataset_raw')
extract_dest.mkdir(parents=True, exist_ok=True)

# Danh sách các thư mục ứng viên trên Google Drive (Ưu tiên thư mục 'dataset' theo Drive của bạn)
drive_candidates = [
    Path('/content/drive/MyDrive/dataset'),
    Path('/content/drive/MyDrive/dataset/MMOTU'),
    Path('/content/drive/MyDrive/colab-U'),
    Path('/content/drive/MyDrive/MMOTU'),
    Path('/content/drive/MyDrive'),
]

found_src_2d = None
found_src_3d = None

if drive_mounted:
    print("🔍 Đang tìm kiếm bộ dữ liệu siêu âm MMOTU trên Google Drive...")
    for cand in drive_candidates:
        if not cand.exists(): continue

        # Kiểm tra trực tiếp
        if (cand / 'OTU_2d').exists() and ((cand / 'OTU_2d' / 'images').exists() or (cand / 'OTU_2d' / 'train_cls.txt').exists()):
            found_src_2d = cand / 'OTU_2d'
            if (cand / 'OTU_3d').exists():
                found_src_3d = cand / 'OTU_3d'
            break

        # Tìm kiếm đệ quy tối đa 2 cấp thư mục (ví dụ dataset/MMOTU/OTU_2d hoặc colab-U/OTU_2d)
        matches_2d = list(cand.glob('*/OTU_2d')) + list(cand.glob('*/*/OTU_2d'))
        if matches_2d:
            found_src_2d = matches_2d[0]
            matches_3d = list(cand.glob('*/OTU_3d')) + list(cand.glob('*/*/OTU_3d'))
            if matches_3d:
                found_src_3d = matches_3d[0]
            break

# Xử lý sao chép dữ liệu sang SSD Colab để tối ưu tốc độ huấn luyện
if found_src_2d:
    print(f"🎯 Đã tìm thấy dữ liệu trên Google Drive tại: {found_src_2d.parent}")
    target_2d = extract_dest / 'OTU_2d'
    target_3d = extract_dest / 'OTU_3d'

    if not (target_2d / 'images').exists():
        print(f"⚡ Đang sao chép OTU_2d sang ổ SSD Colab ({extract_dest}) để tối đa hóa tốc độ đọc GPU...")
        shutil.copytree(found_src_2d, target_2d, dirs_exist_ok=True)
    else:
        print("✅ Dữ liệu OTU_2d đã có sẵn trên SSD Colab!")

    if found_src_3d and not (target_3d / 'images').exists():
        print(f"⚡ Đang sao chép OTU_3d (CEUS) sang ổ SSD Colab ({extract_dest})...")
        shutil.copytree(found_src_3d, target_3d, dirs_exist_ok=True)
    elif found_src_3d:
        print("✅ Dữ liệu OTU_3d đã có sẵn trên SSD Colab!")

    print(f"✅ Bộ dữ liệu MMOTU đã sẵn sàng tại: {extract_dest}")

# Nếu là file nén zip trên Drive
elif drive_mounted and any(p.exists() for p in [
    Path('/content/drive/MyDrive/dataset/MMOTU.zip'),
    Path('/content/drive/MyDrive/dataset.zip'),
    Path('/content/drive/MyDrive/MMOTU.zip')
]):
    zip_p = next(p for p in [
        Path('/content/drive/MyDrive/dataset/MMOTU.zip'),
        Path('/content/drive/MyDrive/dataset.zip'),
        Path('/content/drive/MyDrive/MMOTU.zip')
    ] if p.exists())
    print(f"📦 Tìm thấy file nén {zip_p}. Đang giải nén sang {extract_dest}...")
    shutil.unpack_archive(str(zip_p), str(extract_dest))
    print(f"✅ Giải nén thành công sang {extract_dest}!")

# Kiểm tra thư mục cục bộ (nếu chạy local hoặc đã tải trước)
elif (Path('MMOTU/OTU_2d')).exists():
    extract_dest = Path('MMOTU').resolve()
    print(f"✅ Tìm thấy thư mục MMOTU cục bộ tại: {extract_dest}")
elif (Path('/content/MMOTU/OTU_2d')).exists():
    extract_dest = Path('/content/MMOTU')
    print(f"✅ Tìm thấy thư mục MMOTU tại: {extract_dest}")

# Fallback: Tải từ GitHub nếu chưa có trên Drive
else:
    print("🚀 Không tìm thấy thư mục trên Drive, tự động nạp từ GitHub repository...")
    !git clone https://github.com/son7nui/NCKH-U-buong-trung.git /content/NCKH
    if (Path('/content/NCKH/MMOTU/OTU_2d')).exists():
        shutil.copytree('/content/NCKH/MMOTU/OTU_2d', extract_dest / 'OTU_2d', dirs_exist_ok=True)
        shutil.copytree('/content/NCKH/MMOTU/OTU_3d', extract_dest / 'OTU_3d', dirs_exist_ok=True)
        print(f"✅ Đã nạp thành công dữ liệu từ GitHub về: {extract_dest}")
    else:
        raise FileNotFoundError("Không tìm thấy dữ liệu MMOTU trong MyDrive/dataset, MyDrive/MMOTU hoặc GitHub!")


## 3. Gom Nhóm Ảnh Gần Trùng (pHash), Phân Tầng StratifiedGroupKFold & Lưu Manifest


In [ ]:
RAW_2D = extract_dest / 'OTU_2d'
RAW_3D = extract_dest / 'OTU_3d'

def load_cls(fpath):
    m = {}
    if not os.path.exists(fpath): return m
    with open(fpath, 'r', encoding='utf-8') as f:
        for l in f:
            p = l.strip().split()
            if len(p) >= 2: m[p[0]] = int(p[1])
    return m

# 1. Gom toàn bộ 1.639 ảnh từ 2 phương thức
all_dataset = []
for mod_name, prefix, mod_dir in [('OTU_2d', '2d', RAW_2D), ('OTU_3d', '3d', RAW_3D)]:
    for s in ['train', 'val']:
        cls_f = mod_dir / f"{s}_cls.txt"
        mapping = load_cls(cls_f)
        for img_name, orig_cls in mapping.items():
            base_id = Path(img_name).stem
            img_p = mod_dir / 'images' / img_name
            mask_p = mod_dir / 'annotations' / f"{base_id}_binary.PNG"
            if not mask_p.exists():
                mask_p = mod_dir / 'annotations' / f"{base_id}.PNG"
            if img_p.exists():
                all_dataset.append({
                    'unique_id': f"{prefix}_{base_id}",
                    'img_path': str(img_p),
                    'mask_path': str(mask_p) if mask_p.exists() else None,
                    'orig_cls': orig_cls,
                    'is_tumor': (orig_cls != 5),
                    'modality': prefix,
                    'stratum': f"{prefix}_cls{orig_cls}"
                })

print(f"Tổng số ảnh thực tế nạp được: {len(all_dataset)} ảnh")
assert len(all_dataset) == 1639, f"LỖI: Tổng số ảnh {len(all_dataset)} != 1639!"
unique_ids = set(d['unique_id'] for d in all_dataset)
assert len(unique_ids) == 1639, f"LỖI: Phát hiện trùng lặp mã ID ảnh! ({len(unique_ids)} != 1639)"
print("✅ Kiểm tra dữ liệu: Đầy đủ 1.639 ảnh độc nhất, không trùng lặp ID!")

# 2. Tính Perceptual Hash (64-bit DCT pHash) để phát hiện ảnh gần trùng
def compute_phash_64(img_path, hash_size=8, highfreq_factor=4):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return np.zeros(hash_size * hash_size, dtype=bool)
    resized = cv2.resize(img, (hash_size * highfreq_factor, hash_size * highfreq_factor), interpolation=cv2.INTER_AREA)
    dct = cv2.dct(np.float32(resized))
    dct_low = dct[:hash_size, :hash_size]
    med = np.median(dct_low)
    return (dct_low > med).flatten()

print("\nĐang tính toán Perceptual Hash (pHash) để gom nhóm ảnh gần trùng...")
hashes = [compute_phash_64(d['img_path']) for d in all_dataset]

# Gom nhóm ảnh gần trùng trong cùng một phương thức (Khoảng cách Hamming <= 4)
groups = list(range(len(all_dataset)))
parent = list(range(len(all_dataset)))
def find(i):
    if parent[i] == i: return i
    parent[i] = find(parent[i])
    return parent[i]
def union(i, j):
    root_i = find(i)
    root_j = find(j)
    if root_i != root_j: parent[root_i] = root_j

# So sánh trong từng phương thức
for mod in ['2d', '3d']:
    mod_indices = [idx for idx, d in enumerate(all_dataset) if d['modality'] == mod]
    for idx_a in range(len(mod_indices)):
        i = mod_indices[idx_a]
        h_i = hashes[i]
        for idx_b in range(idx_a + 1, len(mod_indices)):
            j = mod_indices[idx_b]
            if np.count_nonzero(h_i != hashes[j]) <= 4:
                union(i, j)

group_mapping = {}
group_counter = 0
for i in range(len(all_dataset)):
    root = find(i)
    if root not in group_mapping:
        group_mapping[root] = group_counter
        group_counter += 1
    groups[i] = group_mapping[root]

num_clustered = sum(1 for c in Counter(groups).values() if c > 1)
print(f"✅ Phát hiện {num_clustered} cụm ảnh gần trùng (Tổng {len(set(groups))} nhóm độc lập trên 1.639 ảnh).")

# 3. Phân tầng bằng StratifiedGroupKFold hoặc Nạp Manifest đã lưu trên Drive
N_SPLITS = 5
manifest_path = DRIVE_DIR / 'kfold_manifest_v2.csv'
strata = [d['stratum'] for d in all_dataset]

if manifest_path.exists():
    print(f"⏩ Tìm thấy manifest phân chia fold trên Drive ({manifest_path}), đang nạp lại...")
    df_manifest = pd.read_csv(manifest_path)
    id_to_fold = dict(zip(df_manifest['unique_id'], df_manifest['fold']))
    kfold_splits = []
    for f in range(N_SPLITS):
        val_idx = [i for i, d in enumerate(all_dataset) if id_to_fold.get(d['unique_id']) == f]
        train_idx = [i for i, d in enumerate(all_dataset) if id_to_fold.get(d['unique_id']) != f]
        kfold_splits.append((train_idx, val_idx))
    print("✅ Đã nạp thành công 5-Fold split từ Manifest!")
else:
    print("Tạo mới 5-Fold split bằng StratifiedGroupKFold...")
    try:
        sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        kfold_splits = list(sgkf.split(all_dataset, strata, groups=groups))
        print("✅ Phân tầng thành công bằng StratifiedGroupKFold!")
    except Exception as e:
        print(f"⚠️ Cảnh báo StratifiedGroupKFold ({e}), chuyển sang StratifiedKFold chuẩn...")
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        kfold_splits = list(skf.split(all_dataset, strata))

    # Ghi nhận thông tin fold vào dataset và lưu Manifest
    for f_idx, (tr_idx, val_idx) in enumerate(kfold_splits):
        for vi in val_idx: all_dataset[vi]['fold'] = f_idx

    manifest_records = [{
        'unique_id': d['unique_id'],
        'fold': d['fold'],
        'modality': d['modality'],
        'orig_cls': d['orig_cls'],
        'is_tumor': d['is_tumor'],
        'group_id': groups[i]
    } for i, d in enumerate(all_dataset)]
    pd.DataFrame(manifest_records).to_csv(manifest_path, index=False)
    print(f"✅ Đã lưu Manifest phân chia fold vào: {manifest_path}")

# Kiểm tra phân bố từng Fold
print("\nPhân bố mẫu chi tiết trong 5 Folds:")
for f_idx, (tr_idx, val_idx) in enumerate(kfold_splits):
    val_samples = [all_dataset[i] for i in val_idx]
    val_hgs = sum(1 for s in val_samples if s['orig_cls'] == 7)
    val_theca = sum(1 for s in val_samples if s['orig_cls'] == 3)
    val_norm = sum(1 for s in val_samples if s['orig_cls'] == 5)
    val_ceus = sum(1 for s in val_samples if s['modality'] == '3d')
    print(f"  • Fold {f_idx}: Train = {len(tr_idx)} ảnh | Val = {len(val_idx)} ảnh (High-grade={val_hgs}, Theca={val_theca}, Normal={val_norm}, 3D CEUS={val_ceus})")


## 4. Tiền Xử Lý: OCR-Guided Inpainting (en + ch_sim, CEUS riêng), Mask Ratio & Cắt ROI 15% Padding


In [ ]:
# Ánh xạ 7 lớp bệnh học khối u buồng trứng (chuẩn thứ tự bảng chữ cái ImageFolder)
TUMOR_7_CLASSES = {
    0: 'Chocolate_cyst',
    1: 'High_grade_serous',
    2: 'Mucinous_cystadenoma',
    3: 'Serous_cystadenoma',
    4: 'Simple_cyst',
    5: 'Teratoma',
    6: 'Theca_cell_tumor'
}

ORIGINAL_TO_ALPHABETICAL_7CLS = {
    0: 0, # Chocolate_cyst -> 0
    1: 3, # Serous_cystadenoma -> 3
    2: 5, # Teratoma -> 5
    3: 6, # Theca_cell_tumor -> 6
    4: 4, # Simple_cyst -> 4
    6: 2, # Mucinous_cystadenoma -> 2
    7: 1  # High_grade_serous -> 1
}

# 1. Khởi tạo EasyOCR song ngữ Anh - Trung giản thể
print("Khởi tạo EasyOCR song ngữ ['en', 'ch_sim'] trên GPU...")
try:
    import easyocr
    ocr_reader = easyocr.Reader(['en', 'ch_sim'], gpu=torch.cuda.is_available())
    print("✅ Khởi tạo EasyOCR song ngữ thành công!")
    ocr_status = "Available (en + ch_sim)"
except Exception as e:
    ocr_reader = None
    ocr_status = f"Failed ({e})"
    print(f"⚠️ CẢNH BÁO: EasyOCR không khả dụng ({e}). Sẽ sử dụng bộ lọc hình thái học và màu sắc!")

# 2. Hàm khử chú thích phân tách theo phương thức (2D vs CEUS)
def mask_text_and_calipers_ocr_guided(img_bgr, modality='2d', ocr_reader=None, gt_mask=None):
    """
    Khử chú thích văn bản và thước đo siêu âm:
    - Ảnh 2D: Bộ lọc màu vàng HSV + lọc nét mảnh dấu '+', gạch chấm.
    - Ảnh CEUS: Ngưỡng trắng thuần + lọc thành phần liên thông theo diện tích, tỉ lệ khung hình và độ dày nét.
    - OCR song ngữ: Bắt hộp chữ máy in với prob >= 0.40, diện tích và chiều cao tương đối theo kích thước ảnh.
    - Tính toán mask_ratio và gắn cờ cảnh báo.
    """
    img_cleaned = img_bgr.copy()
    h, w = img_bgr.shape[:2]
    combined_mask = np.zeros((h, w), dtype=np.uint8)

    if modality == '2d':
        # Luật Caliper màu vàng cho ảnh 2D thông thường
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        yellow_mask = cv2.inRange(hsv, np.array([18, 100, 100]), np.array([38, 255, 255]))
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(yellow_mask)
        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            if 5 <= area <= 250:
                combined_mask[labels == i] = 255
    else:
        # Luật CEUS riêng: Ngưỡng trắng thuần kết hợp hình thái nét mảnh (không áp dụng luật vàng)
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        white_mask = cv2.inRange(hsv, np.array([0, 0, 230]), np.array([180, 35, 255]))
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(white_mask)
        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            bw = stats[i, cv2.CC_STAT_WIDTH]
            bh = stats[i, cv2.CC_STAT_HEIGHT]
            # Nét chữ/thước đo: diện tích nhỏ, chiều cao < 8% ảnh, aspect ratio thanh mảnh
            if 8 <= area <= 300 and bh < (0.08 * h):
                aspect = bw / float(bh) if bh > 0 else 0
                if 0.15 <= aspect <= 6.0:
                    combined_mask[labels == i] = 255

    # Nhận diện chữ bằng EasyOCR song ngữ
    if ocr_reader is not None:
        try:
            results = ocr_reader.readtext(img_bgr)
            for (bbox, text, prob) in results:
                clean_text = text.strip()
                if prob >= 0.40 and len(clean_text) > 0:
                    pts = np.array(bbox, dtype=np.int32)
                    bx, by, bw, bh = cv2.boundingRect(pts)
                    # Ngưỡng kích thước tương đối theo ảnh
                    if bh < (0.08 * h) and (bw * bh) < (0.03 * h * w):
                        # Kiểm tra độ tương phản trong hộp chữ
                        roi_gray = cv2.cvtColor(img_bgr[max(0, by):min(h, by+bh), max(0, bx):min(w, bx+bw)], cv2.COLOR_BGR2GRAY)
                        if roi_gray.size > 0 and (np.std(roi_gray) > 20.0 or (np.max(roi_gray) - np.min(roi_gray)) > 30):
                            cv2.fillPoly(combined_mask, [pts], 255)
        except Exception:
            pass

    # Tính tỉ lệ mask_ratio (%)
    mask_pixels = np.count_nonzero(combined_mask)
    mask_ratio = (mask_pixels / float(h * w)) * 100.0

    # Kiểm tra cờ cảnh báo
    flag_high_ratio = (mask_ratio > 2.0)
    flag_overlap_tumor = False
    if gt_mask is not None and mask_pixels > 0:
        if np.count_nonzero((combined_mask > 0) & (gt_mask > 0)) > 0:
            flag_overlap_tumor = True

    # Inpaint bằng giải thuật Telea (r=3)
    if mask_pixels > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        dilated_mask = cv2.dilate(combined_mask, kernel, iterations=1)
        img_cleaned = cv2.inpaint(img_cleaned, dilated_mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)

    return img_cleaned, combined_mask, mask_ratio, flag_high_ratio, flag_overlap_tumor

def extract_boxes(mask_path, img_w, img_h, min_area=30):
    """Trích xuất bounding boxes và sắp xếp theo diện tích giảm dần (Đa box GT ưu tiên box lớn nhất)."""
    if not mask_path or not os.path.exists(mask_path): return [], []
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None: return [], []
    thresh = (mask > 0).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    box_candidates = []
    for c in cnts:
        area = cv2.contourArea(c)
        if area < min_area: continue
        x, y, w, h = cv2.boundingRect(c)
        pix_b = (max(0, x), max(0, y), min(img_w, x + w), min(img_h, y + h))
        xc = min(1.0, max(0.0, (x + w / 2.0) / img_w))
        yc = min(1.0, max(0.0, (y + h / 2.0) / img_h))
        nw = min(1.0, max(0.0, w / float(img_w)))
        nh = min(1.0, max(0.0, h / float(img_h)))
        box_candidates.append((area, (xc, yc, nw, nh), pix_b))
    
    # Sắp xếp theo diện tích giảm dần (Largest box first)
    box_candidates.sort(key=lambda item: item[0], reverse=True)
    yolo_boxes = [item[1] for item in box_candidates]
    pix_boxes = [item[2] for item in box_candidates]
    return yolo_boxes, pix_boxes

def crop_roi(img, box, pad=0.15, jitter=False, sz=(300, 300)):
    """Cắt ROI kèm 15% viền mô đệm, hỗ trợ độ lệch ngẫu nhiên BOX_JITTER (±10%)."""
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1

    if jitter:
        # Độ lệch ngẫu nhiên vị trí và kích thước ±10% khi huấn luyện
        shift_x = int(bw * random.uniform(-0.10, 0.10))
        shift_y = int(bh * random.uniform(-0.10, 0.10))
        scale_w = random.uniform(0.90, 1.10)
        scale_h = random.uniform(0.90, 1.10)
        cx, cy = (x1 + x2) / 2.0 + shift_x, (y1 + y2) / 2.0 + shift_y
        bw, bh = bw * scale_w, bh * scale_h
        x1, y1 = cx - bw / 2.0, cy - bh / 2.0
        x2, y2 = cx + bw / 2.0, cy + bh / 2.0

    cx1 = max(0, int(x1 - bw * pad))
    cy1 = max(0, int(y1 - bh * pad))
    cx2 = min(w, int(x2 + bw * pad))
    cy2 = min(h, int(y2 + bh * pad))
    crop = img[cy1:cy2, cx1:cx2]
    if crop.size == 0: return None
    return cv2.resize(crop, sz, interpolation=cv2.INTER_LINEAR)

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    unionArea = float(boxAArea + boxBArea - interArea)
    return interArea / unionArea if unionArea > 0 else 0.0

# 3. Tiền xử lý, tính Mask Ratio & Caching ảnh riêng biệt
CLEAN_CACHE_DIR = Path(f'/content/dataset_{DIR_SUFFIX}')
CLEAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
mask_records = []
flagged_items = []

print(f"\nThực hiện tiền xử lý vào thư mục cache: {CLEAN_CACHE_DIR} (APPLY_INPAINTING={APPLY_INPAINTING})...")
for idx, item in enumerate(all_dataset):
    src_p = Path(item['img_path'])
    dst_p = CLEAN_CACHE_DIR / f"{item['unique_id']}.jpg"
    item['clean_img_path'] = str(dst_p)

    gt_m = None
    if item['mask_path'] and os.path.exists(item['mask_path']):
        gt_m = cv2.imread(item['mask_path'], cv2.IMREAD_GRAYSCALE)

    if not dst_p.exists():
        raw_img = cv2.imread(str(src_p))
        if raw_img is None: continue
        if APPLY_INPAINTING:
            cln_img, c_mask, m_ratio, f_high, f_ovl = mask_text_and_calipers_ocr_guided(
                raw_img, modality=item['modality'], ocr_reader=ocr_reader, gt_mask=gt_m
            )
        else:
            cln_img = raw_img
            m_ratio, f_high, f_ovl = 0.0, False, False
            c_mask = np.zeros(raw_img.shape[:2], dtype=np.uint8)
        
        cv2.imwrite(str(dst_p), cln_img)
    else:
        # Nếu đã có cache, tính nhanh mask_ratio cho báo cáo
        cln_img = cv2.imread(str(dst_p))
        raw_img = cv2.imread(str(src_p))
        diff = cv2.absdiff(raw_img, cln_img)
        m_pixels = np.count_nonzero(cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY) > 10)
        m_ratio = (m_pixels / float(raw_img.shape[0] * raw_img.shape[1])) * 100.0
        f_high = (m_ratio > 2.0)
        f_ovl = (gt_m is not None and np.count_nonzero((diff > 10) & (gt_m[:, :, None] > 0)) > 0)

    item['mask_ratio'] = m_ratio
    item['flag_high_ratio'] = f_high
    item['flag_overlap_tumor'] = f_ovl

    mask_records.append({
        'unique_id': item['unique_id'],
        'modality': item['modality'],
        'orig_cls': item['orig_cls'],
        'mask_ratio': m_ratio,
        'flag_high_ratio': f_high,
        'flag_overlap_tumor': f_ovl
    })
    if f_high or f_ovl:
        flagged_items.append(item)

# Lưu báo cáo mask_ratio lên Drive
df_mask_rep = pd.DataFrame(mask_records)
df_mask_rep.to_csv(DRIVE_DIR / 'mask_ratio_records.csv', index=False)
print("✅ Báo cáo phân bố Mask Ratio theo phương thức:")
for mod in ['2d', '3d']:
    sub_m = df_mask_rep[df_mask_rep['modality'] == mod]['mask_ratio']
    print(f"  • Phương thức {mod.upper()}: Mean={sub_m.mean():.3f}%, Median={sub_m.median():.3f}%, Max={sub_m.max():.2f}%, Số ca gắn cờ={sum((df_mask_rep['modality']==mod) & (df_mask_rep['flag_high_ratio']))}")

# 4. Luôn tạo ảnh QA Panel (kể cả khi cache đã tồn tại)
qa_panel_path = DRIVE_DIR / 'inpainting_qa_panel.png'
random.seed(42)
sample_2d = random.sample([d for d in all_dataset if d['modality'] == '2d'], min(6, len(all_dataset)))
sample_ceus = random.sample([d for d in all_dataset if d['modality'] == '3d'], min(6, len(all_dataset)))
qa_selection = sample_2d + sample_ceus + flagged_items[:12]

qa_rows = []
for it in qa_selection[:18]:
    orig = cv2.imread(it['img_path'])
    cln = cv2.imread(it['clean_img_path'])
    if orig is None or cln is None: continue
    diff = cv2.absdiff(orig, cln)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    mask_vis = (diff_gray > 10).astype(np.uint8) * 255
    mask_c = cv2.applyColorMap(mask_vis, cv2.COLORMAP_HOT)
    overlay = cv2.addWeighted(orig, 0.7, mask_c, 0.3, 0)
    row = np.hstack([cv2.resize(orig, (256, 256)), cv2.resize(overlay, (256, 256)), cv2.resize(cln, (256, 256))])
    qa_rows.append(row)

if qa_rows:
    final_qa = np.vstack(qa_rows)
    cv2.imwrite(str(qa_panel_path), final_qa)
    print(f"✅ Đã tạo ảnh QA Panel kiểm tra Inpainting tại: {qa_panel_path}")


## 5. Vòng Lặp Huấn Luyện & Đánh Giá 5-Fold Không Rò Rỉ (Fixed Training Budget, last.pt, Dual Crops)


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss).clamp(min=1e-6, max=1.0)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

REQUIRED_OOF_COLS = [
    'fold', 'unique_id', 'modality', 'gt_tumor', 'gt_class',
    'screen_tp', 'screen_fp', 'screen_tn', 'screen_fn',
    'loc_tp', 'loc_fp', 'loc_tn', 'loc_fn',
    'top_box_conf', 'max_iou', 'pred_tumor',
    'num_boxes_005', 'num_boxes_030',
    'yolo_crop_pred_class', 'yolo_crop_conf', 'prob_malig',
    'gt_crop_pred_class', 'gt_crop_conf', 'gt_crop_prob_malig',
    'all_probs_yolo', 'all_probs_gt', 'mask_ratio'
]

all_oof_records = []
fold_execution_times = {}

for fold, (train_idx, val_idx) in enumerate(kfold_splits):
    t_fold_start = time.time()
    print("\n" + "=" * 70)
    print(f"🚀 BẮT ĐẦU CHẠY FOLD {fold} / {N_SPLITS - 1}")
    print("=" * 70)

    set_seed(SEED + fold)
    g = torch.Generator()
    g.manual_seed(SEED + fold)

    fold_csv_path = DRIVE_DIR / f"oof_predictions_fold_{fold}_{VERSION_TAG}.csv"
    if fold_csv_path.exists():
        df_existing = pd.read_csv(fold_csv_path)
        missing_cols = [c for c in REQUIRED_OOF_COLS if c not in df_existing.columns]
        if not missing_cols:
            print(f"⏩ [Fold {fold}] Đã có sẵn file OOF v2 hợp lệ trên Drive ({len(df_existing)} mẫu). Nạp kết quả...")
            all_oof_records.extend(df_existing.to_dict('records'))
            continue
        else:
            print(f"⚠️ [Fold {fold}] File OOF cũ thiếu {len(missing_cols)} cột quan trọng: {missing_cols[:4]}... Tiến hành chạy lại!")

    # 1. Thư mục làm việc Fold k
    fold_base = Path(f'/content/dataset_kfold_run_{DIR_SUFFIX}/fold_{fold}')
    yolo_dir = fold_base / 'yolo'
    crops_dir = fold_base / 'crops_7cls'
    yolo_dir.mkdir(parents=True, exist_ok=True)
    crops_dir.mkdir(parents=True, exist_ok=True)

    # 2. Xây dựng dữ liệu Fold k (sử dụng ảnh từ cache sạch)
    for split_tag, indices in [('train', train_idx), ('val', val_idx)]:
        split_yolo_img = yolo_dir / 'images' / split_tag
        split_yolo_lbl = yolo_dir / 'labels' / split_tag
        split_yolo_img.mkdir(parents=True, exist_ok=True)
        split_yolo_lbl.mkdir(parents=True, exist_ok=True)

        for idx in indices:
            item = all_dataset[idx]
            uid = item['unique_id']
            src_clean_img = Path(item['clean_img_path'])
            img = cv2.imread(str(src_clean_img))
            if img is None: continue
            h, w = img.shape[:2]

            dst_img = split_yolo_img / f"{uid}.jpg"
            if not dst_img.exists():
                shutil.copy2(src_clean_img, dst_img)

            y_boxes, p_boxes = extract_boxes(item['mask_path'], w, h)
            with open(split_yolo_lbl / f"{uid}.txt", 'w') as lf:
                if item['orig_cls'] != 5:
                    for xc, yc, bw, bh in y_boxes:
                        lf.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

            # Cắt crop GT cho Stage 2
            if item['orig_cls'] != 5 and item['orig_cls'] in ORIGINAL_TO_ALPHABETICAL_7CLS:
                c_id = ORIGINAL_TO_ALPHABETICAL_7CLS[item['orig_cls']]
                c_name = TUMOR_7_CLASSES[c_id]
                c_dir = crops_dir / split_tag / c_name
                c_dir.mkdir(parents=True, exist_ok=True)

                if p_boxes:
                    for p_i, pbox in enumerate(p_boxes):
                        c_file = c_dir / f"{uid}_crop{p_i}.jpg"
                        if not c_file.exists():
                            # Áp dụng BOX_JITTER khi huấn luyện nếu cờ được bật
                            do_jitter = (split_tag == 'train' and BOX_JITTER is True)
                            roi = crop_roi(img, pbox, pad=0.15, jitter=do_jitter)
                            if roi is not None: cv2.imwrite(str(c_file), roi)
                else:
                    c_file = c_dir / f"{uid}_whole.jpg"
                    if not c_file.exists():
                        w_roi = cv2.resize(img, (300, 300))
                        cv2.imwrite(str(c_file), w_roi)

    with open(yolo_dir / 'dataset.yaml', 'w') as yf:
        yf.write(f"path: {yolo_dir.as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: Tumor\n")

    # 3. Huấn luyện YOLO11n Binary (Fixed 20 Epochs, val=False, seed=SEED+fold, lấy last.pt)
    yolo_local_last = Path(f'/content/runs_kfold/yolo_fold_{fold}_{VERSION_TAG}/weights/last.pt')
    drive_yolo_last = DRIVE_DIR / f'yolo_fold_{fold}_{VERSION_TAG}_last.pt'

    if drive_yolo_last.exists():
        print(f"⏩ [Fold {fold}] Đã có sẵn YOLO last.pt v2 trên Drive! Bỏ qua huấn luyện YOLO...")
        yolo_weight = str(drive_yolo_last)
    elif yolo_local_last.exists():
        print(f"⏩ [Fold {fold}] Đã có sẵn YOLO last.pt v2 cục bộ! Sao lưu lên Drive...")
        shutil.copy2(yolo_local_last, drive_yolo_last)
        yolo_weight = str(yolo_local_last)
    else:
        print(f"\n[Fold {fold}] Huấn luyện YOLO11n Binary (20 epochs cố định, val=False, lấy last.pt)...")
        yolo_model = YOLO('yolo11n.pt')
        yolo_model.train(
            data=(yolo_dir / 'dataset.yaml').as_posix(),
            epochs=20,
            batch=16,
            imgsz=640,
            device=0,
            optimizer='AdamW',
            lr0=0.001,
            val=False, # Không validation để loại trừ triệt để cherry-pick checkpoint
            seed=SEED + fold,
            project='/content/runs_kfold',
            name=f'yolo_fold_{fold}_{VERSION_TAG}',
            plots=False,
            verbose=False
        )
        if yolo_local_last.exists():
            shutil.copy2(yolo_local_last, drive_yolo_last)
            yolo_weight = str(yolo_local_last)
        else:
            yolo_weight = str(Path(f'/content/runs_kfold/yolo_fold_{fold}_{VERSION_TAG}/weights/best.pt'))

    # 4. Huấn luyện EfficientNet-B3 (Fixed 15 Epochs, Focal Loss, Final Checkpoint)
    b3_weight_path = DRIVE_DIR / f'b3_fold_{fold}_{VERSION_TAG}_final.pth'

    train_tf = transforms.Compose([
        transforms.Resize((300, 300)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_tf = transforms.Compose([
        transforms.Resize((300, 300)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_ds = datasets.ImageFolder(crops_dir / 'train', transform=train_tf)
    val_ds = datasets.ImageFolder(crops_dir / 'val', transform=val_tf)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, drop_last=True, worker_init_fn=seed_worker, generator=g)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

    class_names = [TUMOR_7_CLASSES[i] for i in range(7)]
    assert train_ds.classes == class_names, f"LỖI THỨ TỰ NHÃN: {train_ds.classes} != {class_names}"

    model_b3 = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
    in_f = model_b3.classifier[1].in_features
    model_b3.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_f, 512),
        nn.BatchNorm1d(512),
        nn.SiLU(),
        nn.Dropout(p=0.15),
        nn.Linear(512, 7)
    )
    model_b3 = model_b3.to(device)

    if b3_weight_path.exists():
        print(f"⏩ [Fold {fold}] Đã tìm thấy trọng số B3 v2 trên Drive! Bỏ qua huấn luyện B3...")
        model_b3.load_state_dict(torch.load(b3_weight_path, map_location=device))
    else:
        print(f"\n[Fold {fold}] Huấn luyện EfficientNet-B3 (15 epochs cố định, Final Checkpoint)...")
        counts = Counter(train_ds.targets)
        c_weights = [len(train_ds) / (len(counts) * counts[i]) for i in range(len(counts))]
        weights_tensor = torch.FloatTensor(c_weights).to(device)

        criterion = FocalLoss(alpha=weights_tensor, gamma=2.0)
        optimizer = torch.optim.AdamW(model_b3.parameters(), lr=3e-4, weight_decay=1e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

        for epoch in range(1, 16):
            model_b3.train()
            t_loss = 0.0
            for imgs, lbls in train_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                loss = criterion(model_b3(imgs), lbls)
                loss.backward()
                optimizer.step()
                t_loss += loss.item() * imgs.size(0)
            scheduler.step()
            if epoch % 5 == 0 or epoch == 15:
                print(f"  Epoch [{epoch:02d}/15] - Loss: {t_loss / len(train_ds):.4f}")

        torch.save(model_b3.state_dict(), b3_weight_path)
        print(f"✅ Đã lưu checkpoint B3 cuối cùng: {b3_weight_path}")

    # 5. Đánh giá song song toàn diện Fold k
    print(f"\n[Fold {fold}] Đánh giá End-to-End trên {len(val_idx)} ảnh kiểm định...")
    yolo_eval = YOLO(yolo_weight)
    model_b3.eval()
    malig_idx = class_names.index('High_grade_serous')

    fold_records = []
    for idx in val_idx:
        item = all_dataset[idx]
        img_p = item['clean_img_path']
        img_cv = cv2.imread(img_p)
        h, w = img_cv.shape[:2]
        _, gt_pix_boxes = extract_boxes(item['mask_path'], w, h)

        # Suy luận YOLO ở conf=0.05 để thu thập toàn bộ phổ độ tin cậy
        y_preds = yolo_eval.predict(source=img_p, conf=0.05, verbose=False)
        boxes_list = []
        if len(y_preds) > 0 and len(y_preds[0].boxes) > 0:
            for b in y_preds[0].boxes:
                xyxy = b.xyxy[0].cpu().numpy().astype(int)
                conf_score = float(b.conf[0].cpu().numpy())
                boxes_list.append((conf_score, xyxy))
            boxes_list.sort(key=lambda x: x[0], reverse=True)

        num_b_005 = len(boxes_list)
        num_b_030 = sum(1 for b in boxes_list if b[0] >= 0.30)
        top_box_conf = boxes_list[0][0] if len(boxes_list) > 0 else 0.0 # Lưu độ tin cậy thực
        detected_030 = (top_box_conf >= 0.30)
        top_box_coords = boxes_list[0][1] if len(boxes_list) > 0 else None

        # Tính max_iou với các box GT
        max_iou = 0.0
        if top_box_coords is not None and gt_pix_boxes:
            max_iou = max([compute_iou(top_box_coords, gb) for gb in gt_pix_boxes])

        # Mức ảnh (Screening level)
        is_tumor_gt = item['is_tumor']
        screen_tp = (is_tumor_gt is True) and (detected_030 is True)
        screen_fp = (is_tumor_gt is False) and (detected_030 is True)
        screen_tn = (is_tumor_gt is False) and (detected_030 is False)
        screen_fn = (is_tumor_gt is True) and (detected_030 is False)

        # Mức định vị (Localization level, IoU >= 0.50)
        loc_tp = (is_tumor_gt is True) and (max_iou >= 0.50)
        loc_fp = screen_fp or ((is_tumor_gt is True) and detected_030 and (max_iou < 0.50))
        loc_tn = screen_tn
        loc_fn = screen_fn or ((is_tumor_gt is True) and (max_iou < 0.50))

        # Phân loại Stage 2 trên YOLO crop
        yolo_crop_cls = 'Normal_Ovary'
        yolo_crop_conf = 0.0
        prob_malig = 0.0
        probs_yolo_vec = [0.0] * 7
        if detected_030 and top_box_coords is not None:
            roi_yolo = crop_roi(img_cv, top_box_coords, pad=0.15)
            if roi_yolo is not None:
                pil_y = Image.fromarray(cv2.cvtColor(roi_yolo, cv2.COLOR_BGR2RGB))
                inp_y = val_tf(pil_y).unsqueeze(0).to(device)
                with torch.no_grad():
                    logits_y = model_b3(inp_y)
                    p_y = torch.softmax(logits_y, dim=1)[0].cpu().numpy()
                    probs_yolo_vec = [float(val) for val in p_y]
                    top_i = int(np.argmax(p_y))
                    yolo_crop_cls = class_names[top_i]
                    yolo_crop_conf = float(p_y[top_i] * 100.0)
                    prob_malig = float(p_y[malig_idx])

        # Phân loại Stage 2 trên GT crop (chọn box GT lớn nhất)
        gt_crop_cls = 'Normal_Ovary'
        gt_crop_conf = 0.0
        gt_prob_malig = 0.0
        probs_gt_vec = [0.0] * 7
        if is_tumor_gt and gt_pix_boxes:
            # Chú ý: Với ảnh có nhiều box GT, crop GT chọn box có diện tích lớn nhất (primary lesion)
            largest_gt_box = gt_pix_boxes[0]
            roi_gt = crop_roi(img_cv, largest_gt_box, pad=0.15)
            if roi_gt is not None:
                pil_g = Image.fromarray(cv2.cvtColor(roi_gt, cv2.COLOR_BGR2RGB))
                inp_g = val_tf(pil_g).unsqueeze(0).to(device)
                with torch.no_grad():
                    logits_g = model_b3(inp_g)
                    p_g = torch.softmax(logits_g, dim=1)[0].cpu().numpy()
                    probs_gt_vec = [float(val) for val in p_g]
                    top_i_g = int(np.argmax(p_g))
                    gt_crop_cls = class_names[top_i_g]
                    gt_crop_conf = float(p_g[top_i_g] * 100.0)
                    gt_prob_malig = float(p_g[malig_idx])

        gt_cls_name = TUMOR_7_CLASSES[ORIGINAL_TO_ALPHABETICAL_7CLS[item['orig_cls']]] if is_tumor_gt else 'Normal_Ovary'

        rec = {
            'fold': fold,
            'unique_id': item['unique_id'],
            'modality': item['modality'],
            'gt_tumor': is_tumor_gt,
            'gt_class': gt_cls_name,
            'screen_tp': screen_tp,
            'screen_fp': screen_fp,
            'screen_tn': screen_tn,
            'screen_fn': screen_fn,
            'loc_tp': loc_tp,
            'loc_fp': loc_fp,
            'loc_tn': loc_tn,
            'loc_fn': loc_fn,
            'top_box_conf': top_box_conf,
            'max_iou': max_iou,
            'pred_tumor': detected_030,
            'num_boxes_005': num_b_005,
            'num_boxes_030': num_b_030,
            'yolo_crop_pred_class': yolo_crop_cls,
            'yolo_crop_conf': yolo_crop_conf,
            'prob_malig': prob_malig,
            'gt_crop_pred_class': gt_crop_cls,
            'gt_crop_conf': gt_crop_conf,
            'gt_crop_prob_malig': gt_prob_malig,
            'all_probs_yolo': json.dumps(probs_yolo_vec),
            'all_probs_gt': json.dumps(probs_gt_vec),
            'mask_ratio': item.get('mask_ratio', 0.0)
        }
        fold_records.append(rec)
        all_oof_records.append(rec)

    t_fold_end = time.time()
    fold_execution_times[f"fold_{fold}"] = round(t_fold_end - t_fold_start, 2)

    df_fold = pd.DataFrame(fold_records)
    df_fold.to_csv(fold_csv_path, index=False)
    print(f"✅ Hoàn tất Fold {fold}! Đã lưu OOF v2: {fold_csv_path} (Thời gian: {fold_execution_times[f'fold_{fold}']}s)")

# Cập nhật run metadata
meta_path = DRIVE_DIR / 'run_metadata_v2.json'
run_meta = {
    'python_version': sys.version,
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'SEED': SEED,
    'APPLY_INPAINTING': APPLY_INPAINTING,
    'BOX_JITTER': BOX_JITTER,
    'ocr_status': ocr_status,
    'fold_times_seconds': fold_execution_times
}
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(run_meta, f, indent=2, ensure_ascii=False)
print(f"✅ Đã cập nhật metadata thực nghiệm: {meta_path}")


## 6. Báo Cáo Thống Kê Toàn Diện Out-Of-Fold (OOF), Phân Tích Ác Tính, Hiệu Chuẩn & Ma Trận 7x8


In [ ]:
df_oof = pd.DataFrame(all_oof_records)
df_oof.to_csv(DRIVE_DIR / 'full_oof_predictions_1639_v2.csv', index=False)
print(f"Tổng số dự đoán Out-Of-Fold thu được: {len(df_oof)} ảnh / 1.639 ảnh!")

classes_7 = [TUMOR_7_CLASSES[i] for i in range(7)]

def wilson_ci(k, n, z=1.96):
    if n == 0: return "N/A"
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    diff = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return f"{p*100:.2f}% [{max(0, center-diff)*100:.1f}-{min(1, center+diff)*100:.1f}%]"

# 1. BẢNG SÀNG LỌC MỨC ẢNH & ĐỊNH VỊ TỔN THƯƠNG OOF GỘP
def get_screening_row(sub, name):
    tp = sub['screen_tp'].sum()
    fp = sub['screen_fp'].sum()
    tn = sub['screen_tn'].sum()
    fn = sub['screen_fn'].sum()
    tot = len(sub)
    spec_str = wilson_ci(tn, tn + fp) if (tn + fp) > 0 else "N/A (chỉ có mẫu u)"
    return {
        'Phương thức': name, 'Tổng mẫu': tot, 'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
        'Độ nhạy [95% CI]': wilson_ci(tp, tp + fn),
        'Độ đặc hiệu [95% CI]': spec_str,
        'Độ chính xác [95% CI]': wilson_ci(tp + tn, tot)
    }

def get_loc_row(sub, name):
    tp = sub['loc_tp'].sum()
    fp = sub['loc_fp'].sum()
    tn = sub['loc_tn'].sum()
    fn = sub['loc_fn'].sum()
    tot = len(sub)
    spec_str = wilson_ci(tn, tn + fp) if (tn + fp) > 0 else "N/A (chỉ có mẫu u)"
    return {
        'Phương thức': name, 'Tổng mẫu': tot, 'TP (IoU>=0.5)': tp, 'FP': fp, 'TN': tn, 'FN': fn,
        'Độ nhạy định vị [95% CI]': wilson_ci(tp, tp + fn),
        'Độ đặc hiệu [95% CI]': spec_str,
        'Độ chính xác [95% CI]': wilson_ci(tp + tn, tot)
    }

df_screen_sum = pd.DataFrame([
    get_screening_row(df_oof[df_oof['modality'] == '2d'], '2D Thông thường'),
    get_screening_row(df_oof[df_oof['modality'] == '3d'], '3D CEUS Cản âm'),
    get_screening_row(df_oof, 'TỔNG TOÀN BỘ OOF')
])
df_loc_sum = pd.DataFrame([
    get_loc_row(df_oof[df_oof['modality'] == '2d'], '2D Thông thường'),
    get_loc_row(df_oof[df_oof['modality'] == '3d'], '3D CEUS Cản âm'),
    get_loc_row(df_oof, 'TỔNG TOÀN BỘ OOF')
])

print("\n" + "=" * 78)
print("📊 BẢNG 1A: HIỆU NĂNG SÀNG LỌC MỨC ẢNH OOF (KÈM 95% WILSON CI):")
print("=" * 78)
print(df_screen_sum.to_string(index=False))
df_screen_sum.to_csv(DRIVE_DIR / 'summary_screening_oof.csv', index=False)

print("\n" + "=" * 78)
print("📊 BẢNG 1B: HIỆU NĂNG ĐỊNH VỊ TỔN THƯƠNG MỨC IoU >= 0.50 (KÈM 95% WILSON CI):")
print("=" * 78)
print(df_loc_sum.to_string(index=False))
df_loc_sum.to_csv(DRIVE_DIR / 'summary_localization_oof.csv', index=False)

# 2. BẢNG 5 FOLDS & CASCADE DEGRADATION GAP
fold_stats = []
for f in range(N_SPLITS):
    sub_f = df_oof[df_oof['fold'] == f]
    s_tp = sub_f['screen_tp'].sum()
    s_fn = sub_f['screen_fn'].sum()
    s_fp = sub_f['screen_fp'].sum()
    s_tn = sub_f['screen_tn'].sum()
    sens = s_tp / (s_tp + s_fn) if (s_tp + s_fn) > 0 else 0
    spec = s_tn / (s_tn + s_fp) if (s_tn + s_fp) > 0 else 0
    acc = (s_tp + s_tn) / len(sub_f)

    l_tp = sub_f['loc_tp'].sum()
    l_fn = sub_f['loc_fn'].sum()
    loc_sens = l_tp / (l_tp + l_fn) if (l_tp + l_fn) > 0 else 0

    tumors = sub_f[sub_f['gt_tumor'] == True]
    f1_gt = f1_score(tumors['gt_class'], tumors['gt_crop_pred_class'], labels=classes_7, average='macro', zero_division=0)
    f1_yolo = f1_score(tumors['gt_class'], tumors['yolo_crop_pred_class'], labels=classes_7, average='macro', zero_division=0)
    gap = (f1_gt - f1_yolo) * 100.0

    fold_stats.append({
        'Fold': f,
        'Screen_Sens': sens * 100.0,
        'Screen_Spec': spec * 100.0,
        'Screen_Acc': acc * 100.0,
        'Loc_Sens_IoU50': loc_sens * 100.0,
        'MacroF1_GTCrop': f1_gt * 100.0,
        'MacroF1_YOLOCrop': f1_yolo * 100.0,
        'Degradation_Gap': gap
    })

df_fold_summary = pd.DataFrame(fold_stats)
df_fold_summary.to_csv(DRIVE_DIR / 'fold_stats_summary.csv', index=False)
print("\n" + "=" * 80)
print("📊 BẢNG 2: THỐNG KÊ CHI TIẾT 5 FOLDS & CASCADE DEGRADATION GAP:")
print("=" * 80)
print(df_fold_summary.to_string(index=False))
print("-" * 80)
print("Trung bình theo Fold (Mean ± SD):")
print(f"  • Macro-F1 GT Crop: {df_fold_summary['MacroF1_GTCrop'].mean():.2f}% ± {df_fold_summary['MacroF1_GTCrop'].std():.2f}%")
print(f"  • Macro-F1 YOLO Crop: {df_fold_summary['MacroF1_YOLOCrop'].mean():.2f}% ± {df_fold_summary['MacroF1_YOLOCrop'].std():.2f}%")
print(f"  • Cascade Degradation Gap: {df_fold_summary['Degradation_Gap'].mean():.2f}% ± {df_fold_summary['Degradation_Gap'].std():.2f}%")

# Bootstrap 95% CI (B=1000) trên toàn bộ tập OOF gộp
all_tumors_oof = df_oof[df_oof['gt_tumor'] == True].copy()
def bootstrap_gap(df_t, B=1000, seed=42):
    np.random.seed(seed)
    n = len(df_t)
    gt_scores, yolo_scores, gaps = [], [], []
    for _ in range(B):
        sample = df_t.sample(n=n, replace=True)
        m_gt = f1_score(sample['gt_class'], sample['gt_crop_pred_class'], labels=classes_7, average='macro', zero_division=0)
        m_y = f1_score(sample['gt_class'], sample['yolo_crop_pred_class'], labels=classes_7, average='macro', zero_division=0)
        gt_scores.append(m_gt * 100.0)
        yolo_scores.append(m_y * 100.0)
        gaps.append((m_gt - m_y) * 100.0)
    return np.percentile(gt_scores, [2.5, 97.5]), np.percentile(yolo_scores, [2.5, 97.5]), np.percentile(gaps, [2.5, 97.5])

ci_gt, ci_yolo, ci_gap = bootstrap_gap(all_tumors_oof, B=1000)
print("\nToàn bộ OOF Gộp (1.639 ảnh, Bootstrap 95% CI):")
print(f"  • Macro-F1 GT Crop: {f1_score(all_tumors_oof['gt_class'], all_tumors_oof['gt_crop_pred_class'], labels=classes_7, average='macro', zero_division=0)*100:.2f}% [{ci_gt[0]:.2f}% - {ci_gt[1]:.2f}%]")
print(f"  • Macro-F1 YOLO Crop: {f1_score(all_tumors_oof['gt_class'], all_tumors_oof['yolo_crop_pred_class'], labels=classes_7, average='macro', zero_division=0)*100:.2f}% [{ci_yolo[0]:.2f}% - {ci_yolo[1]:.2f}%]")
print(f"  • Cascade Degradation Gap (Δ): {(f1_score(all_tumors_oof['gt_class'], all_tumors_oof['gt_crop_pred_class'], labels=classes_7, average='macro', zero_division=0) - f1_score(all_tumors_oof['gt_class'], all_tumors_oof['yolo_crop_pred_class'], labels=classes_7, average='macro', zero_division=0))*100:.2f}% [{ci_gap[0]:.2f}% - {ci_gap[1]:.2f}%]")

# 3. PHÂN RÃ KẾT QUẢ 7 LỚP THEO PHƯƠNG THỨC (2D VS CEUS)
def get_modality_7cls_table(mod_name):
    sub = all_tumors_oof[all_tumors_oof['modality'] == mod_name]
    rows = []
    for cls in classes_7:
        n_true = sum(sub['gt_class'] == cls)
        tp = sum((sub['gt_class'] == cls) & (sub['yolo_crop_pred_class'] == cls))
        pred_tot = sum(sub['yolo_crop_pred_class'] == cls)
        prec = tp / pred_tot if pred_tot > 0 else 0
        rec = tp / n_true if n_true > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        rows.append({
            'Lớp Bệnh Học': cls, 'Support (N)': n_true,
            'Precision': f"{prec*100:.2f}%", 'Recall': f"{rec*100:.2f}%", 'F1-Score': f"{f1*100:.2f}%"
        })
    return pd.DataFrame(rows)

print("\n" + "=" * 70)
print("📊 BẢNG 3A: PHÂN RÃ 7 LỚP BỆNH HỌC TRÊN SIÊU ÂM 2D (N = 1.469):")
print("=" * 70)
df_2d_cls = get_modality_7cls_table('2d')
print(df_2d_cls.to_string(index=False))
df_2d_cls.to_csv(DRIVE_DIR / 'modality_2d_7class_breakdown.csv', index=False)

print("\n" + "=" * 70)
print("📊 BẢNG 3B: PHÂN RÃ 7 LỚP BỆNH HỌC TRÊN SIÊU ÂM 3D CEUS (N = 170):")
print("=" * 70)
df_ceus_cls = get_modality_7cls_table('3d')
print(df_ceus_cls.to_string(index=False))
df_ceus_cls.to_csv(DRIVE_DIR / 'modality_ceus_7class_breakdown.csv', index=False)

# 4. PHÂN TÍCH BỎ SÓT (FALSE NEGATIVE ANALYSIS) THEO LỚP BỆNH
fn_rows = []
for cls in classes_7:
    sub_c = all_tumors_oof[all_tumors_oof['gt_class'] == cls]
    n_tot = len(sub_c)
    stage1_miss = sum(sub_c['screen_fn'] == True)
    loc_miss = sum((sub_c['loc_tp'] == False))
    fn_rows.append({
        'Lớp Bệnh Học': cls,
        'Số Mẫu Thật (N)': n_tot,
        'Bỏ Sót Stage 1 (FN)': stage1_miss,
        'Tỉ Lệ Bỏ Sót Stage 1': f"{(stage1_miss / n_tot)*100:.2f}%" if n_tot > 0 else "0%",
        'Định Vị Sai/Lệch (IoU<0.5)': loc_miss,
        'Tỉ Lệ Định Vị Lệch': f"{(loc_miss / n_tot)*100:.2f}%" if n_tot > 0 else "0%"
    })
df_fn_analysis = pd.DataFrame(fn_rows)
df_fn_analysis.to_csv(DRIVE_DIR / 'classwise_fn_analysis.csv', index=False)
print("\n" + "=" * 75)
print("📊 BẢNG 4: PHÂN TÍCH BỎ SÓT (FN) THEO TỪNG LỚP BỆNH HỌC:")
print("=" * 75)
print(df_fn_analysis.to_string(index=False))

# 5. PHÂN TÍCH ÁC TÍNH (HIGH-GRADE SEROUS) VS LÀNH TÍNH
y_true_m = (all_tumors_oof['gt_class'] == 'High_grade_serous').astype(int).values
# Score end-to-end: ca bị bỏ sót ở Stage 1 nhận điểm xác suất 0.0
y_scores_yolo = all_tumors_oof['prob_malig'].values
y_scores_gt = all_tumors_oof['gt_crop_prob_malig'].values

auc_yolo = roc_auc_score(y_true_m, y_scores_yolo)
auc_gt = roc_auc_score(y_true_m, y_scores_gt)

# Bootstrap CI cho ROC-AUC
def bootstrap_auc(y_t, y_s, B=1000, seed=42):
    np.random.seed(seed)
    aucs = []
    n = len(y_t)
    for _ in range(B):
        idx = np.random.choice(n, size=n, replace=True)
        if len(np.unique(y_t[idx])) < 2: continue
        aucs.append(roc_auc_score(y_t[idx], y_s[idx]))
    return np.percentile(aucs, [2.5, 97.5])

ci_auc_yolo = bootstrap_auc(y_true_m, y_scores_yolo)
ci_auc_gt = bootstrap_auc(y_true_m, y_scores_gt)

print(f"\n🔬 HIỆU NĂNG PHÂN LOẠI ÁC TÍNH (N = {len(all_tumors_oof)} ca u, {sum(y_true_m)} High-grade serous):")
print(f"  • ROC-AUC trên GT Crop: {auc_gt:.4f} [{ci_auc_gt[0]:.4f} - {ci_auc_gt[1]:.4f}]")
print(f"  • ROC-AUC trên YOLO Crop (End-to-End): {auc_yolo:.4f} [{ci_auc_yolo[0]:.4f} - {ci_auc_yolo[1]:.4f}]")

# Vẽ và lưu đồ thị ROC Ác tính
fpr_gt, tpr_gt, _ = roc_curve(y_true_m, y_scores_gt)
fpr_y, tpr_y, _ = roc_curve(y_true_m, y_scores_yolo)
plt.figure(figsize=(7, 6))
plt.plot(fpr_gt, tpr_gt, color='navy', lw=2, label=f'Stage 2 GT Crop (AUC = {auc_gt:.3f})')
plt.plot(fpr_y, tpr_y, color='darkorange', lw=2, label=f'End-to-End YOLO Crop (AUC = {auc_yolo:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('Tỉ Lệ Dương Tính Giả (False Positive Rate)', fontsize=10)
plt.ylabel('Tỉ Lệ Dương Tính Thật (True Positive Rate)', fontsize=10)
plt.title('Đường Cong ROC Phân Loại Ác Tính (High-Grade Serous vs Lành Tính)', fontsize=11, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_DIR / 'roc_curve_malignancy.png', dpi=300)
plt.close()

# Quét ngưỡng phân loại ác tính 0.1 đến 0.7
sweep_rows = []
for th in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    preds_th = (y_scores_yolo >= th).astype(int)
    tp_m = sum((y_true_m == 1) & (preds_th == 1))
    fp_m = sum((y_true_m == 0) & (preds_th == 1))
    tn_m = sum((y_true_m == 0) & (preds_th == 0))
    fn_m = sum((y_true_m == 1) & (preds_th == 0))
    se = tp_m / (tp_m + fn_m) if (tp_m + fn_m) > 0 else 0
    sp = tn_m / (tn_m + fp_m) if (tn_m + fp_m) > 0 else 0
    ppv = tp_m / (tp_m + fp_m) if (tp_m + fp_m) > 0 else 0
    npv = tn_m / (tn_m + fn_m) if (tn_m + fn_m) > 0 else 0
    f1_m = 2 * ppv * se / (ppv + se) if (ppv + se) > 0 else 0
    sweep_rows.append({
        'Ngưỡng (Threshold)': th,
        'Sensitivity': f"{se*100:.2f}%", 'Specificity': f"{sp*100:.2f}%",
        'PPV': f"{ppv*100:.2f}%", 'NPV': f"{npv*100:.2f}%", 'F1-Score': f"{f1_m*100:.2f}%"
    })
df_sweep = pd.DataFrame(sweep_rows)
df_sweep.to_csv(DRIVE_DIR / 'malignancy_threshold_sweep.csv', index=False)
print("\n" + "=" * 70)
print("📊 BẢNG 5: QUÉT NGƯỠNG PHÂN LOẠI ÁC TÍNH (END-TO-END CASCADE):")
print("=" * 70)
print(df_sweep.to_string(index=False))

# 6. ĐỒ THỊ ROC & PR CHO STAGE 1 (DỰA TRÊN top_box_conf Ở conf=0.05)
y_stage1_gt = df_oof['gt_tumor'].astype(int).values
y_stage1_conf = df_oof['top_box_conf'].values
fpr_s1, tpr_s1, _ = roc_curve(y_stage1_gt, y_stage1_conf)
prec_s1, rec_s1, _ = precision_recall_curve(y_stage1_gt, y_stage1_conf)
auc_s1 = roc_auc_score(y_stage1_gt, y_stage1_conf)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(fpr_s1, tpr_s1, color='blue', lw=2, label=f'Stage 1 ROC (AUC = {auc_s1:.3f})')
# Đánh dấu điểm vận hành 0.30
tp_op = df_oof['screen_tp'].sum()
fp_op = df_oof['screen_fp'].sum()
fn_op = df_oof['screen_fn'].sum()
tn_op = df_oof['screen_tn'].sum()
fpr_op = fp_op / (fp_op + tn_op)
tpr_op = tp_op / (tp_op + fn_op)
ax1.scatter([fpr_op], [tpr_op], color='red', s=80, zorder=5, label=f'Điểm 0.30 (Sens={tpr_op*100:.1f}%, Spec={(1-fpr_op)*100:.1f}%)')
ax1.plot([0, 1], [0, 1], color='gray', linestyle='--')
ax1.set_title('Đường Cong ROC Stage 1 (YOLO11)', fontsize=11, fontweight='bold')
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

ax2.plot(rec_s1, prec_s1, color='green', lw=2, label='Precision-Recall Curve')
prec_op = tp_op / (tp_op + fp_op)
ax2.scatter([tpr_op], [prec_op], color='red', s=80, zorder=5, label=f'Điểm 0.30 (Prec={prec_op*100:.1f}%)')
ax2.set_title('Đường Cong Precision-Recall Stage 1', fontsize=11, fontweight='bold')
ax2.set_xlabel('Recall (Độ nhạy)'); ax2.set_ylabel('Precision (Độ chính xác dương)')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_DIR / 'stage1_roc_pr_curve.png', dpi=300)
plt.close()

# 7. ĐỘ HIỆU CHUẨN MÔ HÌNH (CALIBRATION & RELIABILITY DIAGRAM)
def compute_ece(confidences, predictions_correct, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        b_lo, b_hi = bin_boundaries[i], bin_boundaries[i+1]
        mask = (confidences > b_lo) & (confidences <= b_hi)
        prop = np.mean(mask)
        if prop > 0:
            acc = np.mean(predictions_correct[mask])
            conf = np.mean(confidences[mask])
            ece += np.abs(conf - acc) * prop
            bin_accs.append(acc)
            bin_confs.append(conf)
            bin_counts.append(np.sum(mask))
        else:
            bin_accs.append(0.0); bin_confs.append((b_lo + b_hi)/2); bin_counts.append(0)
    return ece, bin_accs, bin_confs, bin_boundaries

df_det = df_oof[(df_oof['gt_tumor'] == True) & (df_oof['pred_tumor'] == True)]
conf_det = df_det['yolo_crop_conf'].values / 100.0
corr_det = (df_det['gt_class'] == df_det['yolo_crop_pred_class']).astype(int).values
ece_val, b_accs, b_confs, b_bounds = compute_ece(conf_det, corr_det, n_bins=10)
print(f"\n🔬 ĐỘ HIỆU CHUẨN MÔ HÌNH (CALIBRATION): Expected Calibration Error (ECE) = {ece_val:.4f}")

# Vẽ Reliability Diagram
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Hoàn hảo (Perfect Calibration)')
mid_points = (b_bounds[:-1] + b_bounds[1:]) / 2.0
plt.bar(mid_points, b_accs, width=0.08, alpha=0.5, color='royalblue', edgecolor='navy', label='Độ chính xác thực tế')
plt.scatter(mid_points, b_confs, color='red', s=40, label='Độ tin cậy dự đoán')
plt.xlabel('Độ Tin Cậy Dự Đoán (Confidence)', fontsize=10)
plt.ylabel('Độ Chính Xác Thực Tế (Accuracy)', fontsize=10)
plt.title(f'Reliability Diagram (ECE = {ece_val:.4f})', fontsize=11, fontweight='bold')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_DIR / 'calibration_reliability_diagram.png', dpi=300)
plt.close()

# 8. MA TRẬN NHẦM LẪN CHUẨN HÓA (7x7 VÀ 7x8)
# A. Ma trận 7x7 Có điều kiện
cm_cond = confusion_matrix(df_det['gt_class'], df_det['yolo_crop_pred_class'], labels=classes_7, normalize='true')
plt.figure(figsize=(8.5, 7))
sns.heatmap(cm_cond, annot=True, fmt='.2f', cmap='Blues', xticklabels=classes_7, yticklabels=classes_7)
plt.title('Ma Trận 7x7 Có Điều Kiện (Conditional on Stage 1 Detection)', fontsize=11, fontweight='bold')
plt.xlabel('Nhãn Dự Đoán Stage 2', fontsize=10)
plt.ylabel('Nhãn Thật (Ground-Truth)', fontsize=10)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(DRIVE_DIR / 'stage2_confusion_matrix_7cls_conditional.png', dpi=300)
plt.close()

# B. Ma trận 7x8 End-to-End Cascade (kèm số lượng mẫu hàng)
cols_8 = classes_7 + ['Missed_by_Stage1']
all_tumors_oof['pred_col_8'] = all_tumors_oof['yolo_crop_pred_class'].apply(lambda x: x if x in classes_7 else 'Missed_by_Stage1')
cm_8 = confusion_matrix(all_tumors_oof['gt_class'], all_tumors_oof['pred_col_8'], labels=cols_8)[:7, :]
row_sums = cm_8.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
cm_8_norm = cm_8 / row_sums

# Tên hàng kèm số lượng mẫu
row_labels_with_n = [f"{cls} (N={sum(all_tumors_oof['gt_class'] == cls)})" for cls in classes_7]

plt.figure(figsize=(10.5, 7.5))
sns.heatmap(cm_8_norm, annot=True, fmt='.2f', cmap='Reds', xticklabels=cols_8, yticklabels=row_labels_with_n)
plt.title('Ma Trận Nhầm Lẫn 7x8 End-to-End Toàn Bộ Pipeline (Bao Gồm Cột Bị Bỏ Sót)', fontsize=11, fontweight='bold')
plt.xlabel('Nhãn Dự Đoán End-to-End (Cột 8: Bỏ Sót Ở Stage 1)', fontsize=10)
plt.ylabel('Nhãn Thật (Kèm Cỡ Mẫu N)', fontsize=10)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(DRIVE_DIR / 'oof_confusion_matrix_7x8_end_to_end.png', dpi=300)
plt.close()

print("✅ Đã xuất bản toàn bộ biểu đồ, đồ thị ROC và ma trận nhầm lẫn chuẩn khoa học lên Drive!")


## 7. Explainable AI (Grad-CAM) Khách Quan: Đối Chiếu Đúng Fold, Sửa Màu RGB & Định Lượng Năng Lượng


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hooks = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, inp, out):
            self.activations = out
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]
        self.hooks.append(self.target_layer.register_forward_hook(forward_hook))
        self.hooks.append(self.target_layer.register_full_backward_hook(backward_hook))

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        score = output[0, target_class]
        score.backward(retain_graph=True)
        
        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().detach().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, output

    def remove_hooks(self):
        for h in self.hooks: h.remove()

# 1. Chọn các ca có chủ đích từ OOF
curated_cases = []
# Ca 1: TP High-grade serous
c_tp_hgs = df_oof[(df_oof['gt_class'] == 'High_grade_serous') & (df_oof['yolo_crop_pred_class'] == 'High_grade_serous')]
if len(c_tp_hgs) > 0: curated_cases.append(('TP High-grade Serous', c_tp_hgs.sample(1, random_state=42).iloc[0]))

# Ca 2: TP Teratoma
c_tp_tera = df_oof[(df_oof['gt_class'] == 'Teratoma') & (df_oof['yolo_crop_pred_class'] == 'Teratoma')]
if len(c_tp_tera) > 0: curated_cases.append(('TP Teratoma', c_tp_tera.sample(1, random_state=42).iloc[0]))

# Ca 3: FP trên Buồng trứng bình thường (Normal)
c_fp_norm = df_oof[(df_oof['gt_tumor'] == False) & (df_oof['pred_tumor'] == True)]
if len(c_fp_norm) > 0: curated_cases.append(('FP Normal Ovary', c_fp_norm.sample(1, random_state=42).iloc[0]))

# Ca 4: FN Bỏ sót ở Stage 1 (dùng GT Crop)
c_fn_miss = df_oof[(df_oof['gt_class'] == 'High_grade_serous') & (df_oof['pred_tumor'] == False)]
if len(c_fn_miss) > 0:
    curated_cases.append(('FN Missed by Stage 1 [Dùng GT Crop]', c_fn_miss.sample(1, random_state=42).iloc[0]))

# Ca 5: High-grade serous bị phân loại sai
c_mis_hgs = df_oof[(df_oof['gt_class'] == 'High_grade_serous') & (df_oof['pred_tumor'] == True) & (df_oof['yolo_crop_pred_class'] != 'High_grade_serous')]
if len(c_mis_hgs) > 0: curated_cases.append(('Misclassified High-grade', c_mis_hgs.sample(1, random_state=42).iloc[0]))

# Ca 6: Ca có mask_ratio cao
c_high_mask = df_oof.sort_values(by='mask_ratio', ascending=False)
if len(c_high_mask) > 0: curated_cases.append(('High Mask Ratio', c_high_mask.iloc[0]))

print(f"✅ Đã chọn {len(curated_cases)} ca kiểm tra Grad-CAM có chủ đích.")

# 2. Cache các mô hình B3 theo fold để nạp đúng mô hình độc lập
b3_models_cache = {}
def get_b3_fold_model(fold_id):
    if fold_id not in b3_models_cache:
        w_p = DRIVE_DIR / f'b3_fold_{fold_id}_{VERSION_TAG}_final.pth'
        if not w_p.exists():
            w_p = DRIVE_DIR / f'b3_fold_{fold_id}_final.pth' # fallback nếu có
        m = efficientnet_b3(weights=None)
        in_f = m.classifier[1].in_features
        m.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_f, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(p=0.15),
            nn.Linear(512, 7)
        )
        if w_p.exists():
            m.load_state_dict(torch.load(w_p, map_location=device))
        m = m.to(device)
        m.eval()
        b3_models_cache[fold_id] = m
    return b3_models_cache[fold_id]

# 3. Trực quan hóa 4 cột & Định lượng năng lượng Heatmap
val_tf_cam = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

fig, axes = plt.subplots(len(curated_cases), 4, figsize=(15, len(curated_cases) * 3.5))
if len(curated_cases) == 1: axes = np.expand_dims(axes, 0)

energy_ratios = []

for r_idx, (group_name, rec) in enumerate(curated_cases):
    case_fold = int(rec['fold'])
    model_for_case = get_b3_fold_model(case_fold)
    gradcam_inst = GradCAM(model_for_case, model_for_case.features[-1])

    # Tìm lại item từ all_dataset
    matching = [d for d in all_dataset if d['unique_id'] == rec['unique_id']]
    raw_path = matching[0]['img_path'] if matching else rec['clean_img_path']
    clean_path = matching[0]['clean_img_path'] if matching else rec['clean_img_path']

    raw_bgr = cv2.imread(raw_path)
    clean_bgr = cv2.imread(clean_path)
    if raw_bgr is None or clean_bgr is None: continue

    h, w = raw_bgr.shape[:2]
    _, gt_boxes = extract_boxes(matching[0]['mask_path'], w, h) if matching else ([], [])

    # Quyết định box ROI: nếu là ca FN missed thì dùng GT box lớn nhất, ngược lại dùng YOLO box
    is_fn_case = ('FN' in group_name)
    if is_fn_case and gt_boxes:
        eval_box = gt_boxes[0]
    elif rec['top_box_conf'] >= 0.30 and gt_boxes:
        eval_box = gt_boxes[0]
    else:
        eval_box = (0, 0, w, h)

    roi_crop_bgr = crop_roi(clean_bgr, eval_box, pad=0.15)
    if roi_crop_bgr is None: roi_crop_bgr = cv2.resize(clean_bgr, (300, 300))

    # Chuyển BGR sang RGB cho crop trước khi tính toán/hiển thị
    roi_crop_rgb = cv2.cvtColor(roi_crop_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(roi_crop_rgb)
    inp = val_tf_cam(pil_img).unsqueeze(0).to(device)

    cam_mask, out_logits = gradcam_inst.generate(inp)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_mask), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Pha trộn màu sắc chuẩn RGB (Khắc phục lỗi màu ở bản cũ)
    overlay = 0.5 * (heatmap_rgb / 255.0) + 0.5 * (roi_crop_rgb / 255.0)
    overlay = overlay / np.max(overlay)

    # Cột 1: Ảnh gốc với bounding box (trước inpaint)
    vis_raw = cv2.cvtColor(raw_bgr.copy(), cv2.COLOR_BGR2RGB)
    if gt_boxes:
        cv2.rectangle(vis_raw, (gt_boxes[0][0], gt_boxes[0][1]), (gt_boxes[0][2], gt_boxes[0][3]), (255, 0, 0), 2)
    axes[r_idx, 0].imshow(vis_raw)
    axes[r_idx, 0].set_title(f"Ảnh Gốc (Box GT) | {group_name}", fontsize=9, fontweight='bold')
    axes[r_idx, 0].axis('off')

    # Cột 2: Ảnh sau Inpaint với Crop Box
    vis_cln = cv2.cvtColor(clean_bgr.copy(), cv2.COLOR_BGR2RGB)
    cv2.rectangle(vis_cln, (eval_box[0], eval_box[1]), (eval_box[2], eval_box[3]), (0, 255, 0), 2)
    axes[r_idx, 1].imshow(vis_cln)
    axes[r_idx, 1].set_title(f"Sau Inpaint (Mask={rec.get('mask_ratio', 0):.2f}%)", fontsize=9)
    axes[r_idx, 1].axis('off')

    # Cột 3: Grad-CAM Overlay
    axes[r_idx, 2].imshow(overlay)
    axes[r_idx, 2].set_title(f"Grad-CAM (Fold {case_fold} Model)", fontsize=9, fontweight='bold')
    axes[r_idx, 2].axis('off')

    # Cột 4: Biểu đồ xác suất 7 lớp
    probs = torch.softmax(out_logits, dim=1)[0].cpu().detach().numpy()
    pred_top = classes_7[np.argmax(probs)]
    axes[r_idx, 3].barh(classes_7, probs, color='skyblue')
    axes[r_idx, 3].set_xlim(0, 1.0)
    axes[r_idx, 3].set_title(f"GT: {rec['gt_class']}\nPred: {pred_top} ({np.max(probs)*100:.1f}%)", fontsize=8)
    axes[r_idx, 3].tick_params(axis='y', labelsize=8)

    gradcam_inst.remove_hooks()

plt.tight_layout()
gradcam_fig_path = DRIVE_DIR / 'gradcam_explainable_ai_curated.png'
plt.savefig(gradcam_fig_path, dpi=300)
plt.close()
print(f"✅ Đã tạo và lưu thành công trực quan hóa Grad-CAM chuẩn y khoa tại: {gradcam_fig_path}")


## 8. Khung Thử Nghiệm Đối Chứng (Ablation Study Framework)


In [ ]:
# CỜ ĐIỀU KHIỂN CHẠY ABLATION (Mặc định False để không làm gián đoạn luồng chính)
# Chuyển RUN_ABLATION = True khi bạn muốn chạy thực nghiệm đối chứng 3 trục:
# 1. Có vs Không Inpainting (APPLY_INPAINTING: True vs False)
# 2. Focal Loss vs Cross-Entropy chuẩn
# 3. Có vs Không mở rộng 15% padding viền mô đệm

if RUN_ABLATION:
    print("🔬 BẮT ĐẦU CHẠY THỬ NGHIỆM ĐỐI CHỨNG (ABLATION STUDY)...")
    ablation_configs = [
        {'name': 'Baseline_Inpaint_Focal_Pad15', 'inpaint': True, 'loss': 'focal', 'pad': 0.15},
        {'name': 'Ablation_NoInpaint_Focal_Pad15', 'inpaint': False, 'loss': 'focal', 'pad': 0.15},
        {'name': 'Ablation_Inpaint_CE_Pad15', 'inpaint': True, 'loss': 'ce', 'pad': 0.15},
        {'name': 'Ablation_Inpaint_Focal_NoPad', 'inpaint': True, 'loss': 'focal', 'pad': 0.0}
    ]
    print(f"Đã chuẩn bị sẵn {len(ablation_configs)} cấu hình thử nghiệm so ghép cặp.")
    # (Khung code đã sẵn sàng để người dùng kích hoạt khi cần xuất bản bài báo chuyên sâu)
else:
    print("ℹ️ Cờ RUN_ABLATION = False. Bỏ qua bước huấn luyện ablation tự động. Khung thử nghiệm đã sẵn sàng!")


## 9. Hạn Chế Của Nghiên Cứu (Study Limitations)

Nhằm đảm bảo tính nghiêm cẩn khoa học và tính minh bạch trong báo cáo y khoa, nghiên cứu ghi nhận các hạn chế phương pháp luận sau:
1. **Thiếu Patient ID & Rủi Ro Rò Rỉ Cùng Bệnh Nhân**: Tập dữ liệu công khai MMOTU không cung cấp mã định danh bệnh nhân (*Patient ID*). Dù nhóm nghiên cứu đã áp dụng kỹ thuật băm cảm nhận (*Perceptual Hash - pHash*) để gom nhóm các ảnh gần trùng và chia phân tầng bằng `StratifiedGroupKFold`, vẫn tồn tại rủi ro các lát cắt siêu âm từ cùng một ca bệnh nhân rơi vào cả tập huấn luyện và kiểm định.
2. **Ảnh Hưởng Của Việc Tinh Chỉnh Siêu Tham Số Ban Đầu**: Một số tham số thiết kế (ngưỡng phát hiện 0.30, 15% padding viền mô đệm, các ngưỡng lọc diện tích OCR/HSV) từng được khảo sát và tinh chỉnh trên tập validation chính thức của MMOTU trước khi tiến hành chia 5-Fold xoay vòng.
3. **Mẫu U Ác Tính Còn Khiêm Tốn**: Toàn bộ kho dữ liệu chỉ có 62 ca ung thư biểu mô (*High-grade serous carcinoma*) trên tổng số 1.639 ảnh ($3.78\%$). Sự mất cân bằng mẫu tự nhiên này dẫn đến khoảng tin cậy của phân loại ác tính còn tương đối rộng.
4. **Nghiên Cứu Đơn Trung Tâm**: Dữ liệu được thu thập từ một trung tâm y tế duy nhất với cùng thế hệ máy siêu âm. Cần kiểm chứng tiền cứu đa trung tâm độc lập (*External Multi-Center Validation*) để thẩm định năng lực tổng quát hóa trong thực tế lâm sàng.
5. **Bản Chất Vật Lý Của Siêu Âm 3D CEUS**: Tập con OTU_3d là các lát cắt từ siêu âm cản âm vi mạch (*Contrast-Enhanced Ultrasound*), phản ánh động học tưới máu chứ không phải hình ảnh B-mode 2D giải phẫu đơn thuần.
6. **Chưa Tích Hợp Hợp Nhất Đa Phương Thức Sâu**: Hệ thống hiện tại huấn luyện mô hình chung trên dữ liệu hỗn hợp 2D và 3D CEUS, chưa xây dựng kiến trúc mạng nhánh đôi (*Dual-branch network*) để hợp nhất đặc trưng tương hỗ giữa hai phương thức.
